# S2G Interactive Notebook
This notebook allows you to play around with the S2G model inputs, test out different schemas, and see the extracted knowledge graph in real time.

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from s2g.linearisation import S2GTokens, add_special_tokens_to_tokenizer
from s2g.scripts.config_utils import load_schema, load_entity_schema
from s2g.scripts.inference import extract_re, extract_boundary_joint

## Initialization
Load the model, tokenizer, and define schemas. Change the `checkpoint` variable to point to your trained model path or use `google/flan-t5-small` to test the untuned model.

In [ ]:
checkpoint = "google/flan-t5-small" # Replace with your trained checkpoint path
model_variant = "joint" # Change to "re", "boundary_re", "joint", or "boundary_joint"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)

# Add S2G special tokens
tokens = S2GTokens(model_variant, use_rejection=False)
add_special_tokens_to_tokenizer(tokenizer, tokens, model)

## Define Schemas
You can either load schemas from a file, or define them manually as lists of strings.

In [ ]:
# Optionally load from file: 
# entity_schema = load_entity_schema("configs/schema/conll04_entities.json")
# rel_schema = load_schema("configs/schema/conll04_relations.json")

entity_schema = ["Loc", "Org", "Peop", "Other"]
rel_schema = ["Work_For", "Kill", "Organization_Based_In", "Live_In", "Located_In"]

## Inference
Run inference on a piece of text. We wrap the inference function based on the model variant.

In [ ]:
text = "John Wilkes Booth, who assassinated President Abraham Lincoln, was an actor from Maryland."

extract_fn = extract_re if model_variant in {"re", "boundary_re"} else extract_boundary_joint
tasks = [model_variant] # e.g. ["joint"]

result = extract_fn(
    text=text,
    model=model,
    tokenizer=tokenizer,
    entity_schema=entity_schema,
    rel_schema=rel_schema,
    tokens=tokens,
    num_beams=4,
    max_source_length=300,
    max_target_length=200,
    device=device,
    constraint_decoding=False,
    tasks=tasks,
    ssi_prompt="ssi" # Can be "ssi" or "natural"
)

print(json.dumps(result, indent=2))

## Raw Prompting
If you want to experiment with the model directly (providing the exact encoder string and viewing the raw string output without constraint decoding or parsing), use the cell below.

In [ ]:
raw_input = "<text> John Wilkes Booth, who assassinated President Abraham Lincoln, was an actor from Maryland. </text> <re> <head> <type> <rel> <tail> <type> <nest> "

tok_out = tokenizer([
    raw_input
], max_length=300, truncation=True, return_tensors="pt").to(device)

with torch.inference_mode():
    generated = model.generate(
        **tok_out, 
        num_beams=4, 
        max_length=200
    )

raw_output = tokenizer.decode(generated[0], skip_special_tokens=False)
print("Model Output:")
print(raw_output)